In [ ]:
# Install deps if needed (run once)
import subprocess, sys
try:
    import mlx_tune, datasets, dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx-tune", "datasets", "python-dotenv", "openai", "groq", "requests", "beautifulsoup4", "lxml", "tqdm"])

/Users/vyakaranamsowmya/Desktop/idk/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load API keys from .env file
import os
from dotenv import load_dotenv
load_dotenv()

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
HUGGINGFACE_HUB_TOKEN = os.getenv('HUGGINGFACE_HUB_TOKEN')

print('DEEPSEEK_API_KEY set:', bool(DEEPSEEK_API_KEY))
print('HUGGINGFACE_HUB_TOKEN set:', bool(HUGGINGFACE_HUB_TOKEN))


DEEPSEEK_API_KEY set: True
HUGGINGFACE_HUB_TOKEN set: True


## Phase 2: Generate Training Data (AutoDidact)

Generates 10k+ Q&A pairs grounded in real regulatory documents (PMLA 2002, FIU-IND enforcement orders, SEBI reports).
Uses Groq llama-3.3-70b as teacher LLM — reads source chunks, generates natural exam-style Q&A.


## Phase 2b: AutoDidact Generation (FIU-IND + SEBI grounded)

Generates 10k Q&A pairs grounded in real regulatory documents (PMLA 2002, FIU-IND enforcement orders, SEBI reports) using Groq llama-3.3-70b as teacher LLM.


In [ ]:
# Phase 2b NOTE: Data generation now runs OUTSIDE the notebook.
# DeepSeek writes to data/fintech_data_grounded.json via:
#   AUTODIDACT_TARGET=15000 .venv/bin/python3.11 fintech_data/autodidact_gen.py
# The old Groq/AutoDidactFetcher cell was removed (obsolete + would corrupt files).
# Just confirm the generated data exists:

import json, os
grounded_path = os.path.join("data", "fintech_data_grounded.json")
if os.path.exists(grounded_path):
    with open(grounded_path, encoding="utf-8") as f:
        grounded = json.load(f)
    print(f"DeepSeek-grounded pairs available: {len(grounded)}")
else:
    print("No grounded data yet - run the DeepSeek job first.")


DeepSeek-grounded pairs available: 15001


## Phase 4: Process & Merge Data

Combine all data sources, filter quality, split into train/eval, format for Qwen chat template.


In [ ]:
from fintech_data import DataProcessor
import os, json, random

if 'processor' not in globals():
    processor = DataProcessor()

# 1) Load DeepSeek-grounded QA pairs (pure generated, source=autodidact)
grounded_pairs = []
for grounded_path in [
    os.path.join("data", "fintech_data_grounded.json"),
    os.path.join("data", "fintech_data_grounded_rbi.json"),
]:
    if os.path.exists(grounded_path):
        with open(grounded_path, encoding="utf-8") as f:
            grounded_pairs.extend(json.load(f))
        print(f"Loaded grounded (DeepSeek): {os.path.basename(grounded_path)} -> {len(grounded_pairs)} pairs")
    else:
        print(f"No grounded data yet - run the DeepSeek job first: {grounded_path}")

# 2) Load HF dataset pairs (Priyanshu-24 adaption)
autodidact_path = os.path.join("data", "fintech_data_autodidact.json")
autodidact_pairs = []
if os.path.exists(autodidact_path):
    with open(autodidact_path, encoding="utf-8") as f:
        autodidact_pairs = json.load(f)
    print(f"Loaded autodidact (HF): {len(autodidact_pairs)} pairs")

# 3) Load the reliable base dataset (HF finance QA + RBI master direction PDFs)
reliable_path = os.path.join("data", "fintech_data_reliable.json")
reliable_pairs = []
if os.path.exists(reliable_path):
    with open(reliable_path, encoding="utf-8") as f:
        reliable_pairs = json.load(f)
    print(f"Loaded reliable base: {len(reliable_pairs)} pairs")

# Balance sources: cap dominant groups, keep all from smaller sources
def balance(pairs, cap):
    random.Random(42).shuffle(pairs)
    return pairs[:cap]

balanced = (
    balance(grounded_pairs, 999999)
    + balance(autodidact_pairs, 8000)
    + balance(reliable_pairs, 12000)
)
print(f"Balanced pool: {len(balanced)} pairs (grounded kept fully at {len(grounded_pairs)})")

# Merge + quality filter
all_pairs = balanced
all_pairs = processor.filter_quality(all_pairs)
print(f"Total pairs after merge: {len(all_pairs)}")

formatted = processor.format_for_training(all_pairs, model_type="qwen")
train_data, eval_data = processor.split_dataset(formatted, train_ratio=0.85)
processor.save_split(train_data, eval_data)


Loaded grounded (DeepSeek): fintech_data_grounded.json -> 15001 pairs
Loaded grounded (DeepSeek): fintech_data_grounded_rbi.json -> 23251 pairs
Loaded autodidact (HF): 7750 pairs
Loaded reliable base: 15417 pairs
Balanced pool: 43001 pairs (grounded kept fully at 23251)
Quality filter: 43001 -> 40378 kept
Total pairs after merge: 40378
Train: 34321, Eval: 6057
Saved train (34321) and eval (6057)


(PosixPath('data/fintech_train.json'), PosixPath('data/fintech_eval.json'))

## Phase 5: LoRA Fine-Tuning

Applies LoRA to Qwen2.5-1.5B. Only ~0.5% of parameters are trained (adapter weights).

In [ ]:
from fintech_data import FintechFineTuner, DataProcessor

if 'processor' not in globals():
    processor = DataProcessor()

train_data, eval_data = processor.load_dataset()
print(f"Loaded: {len(train_data)} train, {len(eval_data)} eval")

if 'model_name' not in globals():
    model_name = "mlx-community/Qwen2.5-1.5B-4bit"

tuner = FintechFineTuner(
    base_model_name=model_name,
    output_dir="fintech_finetuned_qwen",
    max_seq_length=1024,
)

tuner.load_base_model()
tuner.apply_lora(r=16, lora_alpha=32)

tuner.train(
    train_data=train_data,
    eval_data=eval_data,
    num_epochs=3,
    batch_size=2,
    learning_rate=2e-4,
    gradient_accumulation_steps=4,
    max_seq_length=1024,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
)


Loaded: 34321 train, 6057 eval
Loading base model: mlx-community/Qwen2.5-1.5B-4bit


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1262.08it/s]


Base model loaded
LoRA configuration set: rank=16, alpha=32, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], dropout=0.05
LoRA applied (r=16, alpha=32)


/Users/vyakaranamsowmya/Desktop/idk/.venv/lib/python3.11/site-packages/mlx_tune/model.py:337: UserWarning: LoRA dropout may have limited support in MLX. Dropout value will be set but behavior may differ from PyTorch PEFT.
  warnings.warn(


Trainer initialized:
  Output dir: fintech_finetuned_qwen
  Adapter path: fintech_finetuned_qwen/adapters
  Learning rate: 0.0002
  Iterations: 51480
  Batch size: 2
  LoRA r=16, alpha=32
  Native training: True
  LR scheduler: cosine
  Grad checkpoint: False
Starting training...
Starting Fine-Tuning

[Using Native MLX Training]

Applying LoRA adapters...
Applying LoRA to 28 layers: {'rank': 16, 'scale': 2.0, 'dropout': 0.05, 'keys': ['mlp.down_proj', 'mlp.gate_proj', 'mlp.up_proj', 'self_attn.k_proj', 'self_attn.o_proj', 'self_attn.q_proj', 'self_attn.v_proj']}
✓ LoRA applied successfully to 28 layers
  Trainable LoRA parameters: 392
Preparing training data...
  Detected format: text
✓ Prepared 34321 training samples
  Saved to: fintech_finetuned_qwen/train.jsonl
✓ Prepared validation set

Training configuration:
  Iterations: 51480
  Batch size: 2
  Learning rate: 0.0002
  LR scheduler: cosine
  Grad checkpoint: False
  Adapter file: fintech_finetuned_qwen/adapters/adapters.safetenso

...

Calculating loss...:  12%|█▏        | 24/200 [01:42<17:02,  5.81s/it]

[WARNING] Some sequences are longer than 1024 tokens. The longest sentence 1223 will be truncated to 1024. Consider pre-splitting your data to save memory.


Calculating loss...:  15%|█▌        | 30/200 [02:09<14:39,  5.18s/it]

[WARNING] Some sequences are longer than 1024 tokens. The longest sentence 1032 will be truncated to 1024. Consider pre-splitting your data to save memory.


Calculating loss...:  17%|█▋        | 34/200 [02:24<08:57,  3.24s/it]

## Phase 6: Test the Fine-Tuned Model

In [ ]:
# Test the Colab-trained fine-tuned model
# The adapter was trained on Google Colab (HF PEFT format) and downloaded to:
#   ~/Downloads/finlens_adapter_final
# This cell converts it to MLX format, loads it, and runs the test questions.
import os
import shutil
from pathlib import Path

from convert_peft_to_mlx import convert
from fintech_data.finetune import FintechFineTuner

HF_ADAPTER = Path.home() / "Downloads" / "finlens_adapter_final"
MLX_ADAPTER = Path("fintech_finetuned_qwen/adapters_colab")

assert HF_ADAPTER.exists(), f"adapter not found at {HF_ADAPTER}"

# 1) Convert Hugging Face PEFT adapter -> MLX format
if MLX_ADAPTER.exists():
    shutil.rmtree(MLX_ADAPTER)
convert(str(HF_ADAPTER), str(MLX_ADAPTER))

# 2) Load the converted adapter
tuner = FintechFineTuner(max_seq_length=512)
tuner.load_for_inference(str(MLX_ADAPTER))

# 3) Run the test questions
test_questions = [
    "What is CKYC and how is it different from regular KYC?",
    "What are the steps for CERSAI registration?",
    "What is the difference between VKYC and e-KYC?",
    "What is AML and what are the reporting obligations for a fintech?",
    "How does the PMLA 2002 affect financial institutions?",
]

for q in test_questions:
    print("\n" + "=" * 60)
    print(f"Q: {q}")
    print("=" * 60)
    response = tuner.inference(q)
    print(f"A: {response}")
